In [1]:
import sqlite3
import pandas as pd
import random
from datetime import datetime, timedelta

# Create DB
conn = sqlite3.connect("asset_management.db")
cursor = conn.cursor()

In [2]:
# CREATE TABLES

cursor.executescript("""
DROP TABLE IF EXISTS Assets;
DROP TABLE IF EXISTS Maintenance;
DROP TABLE IF EXISTS Inspections;
DROP TABLE IF EXISTS Outages;

CREATE TABLE Assets (
    asset_id INTEGER PRIMARY KEY,
    asset_type TEXT,
    location TEXT,
    install_date DATE,
    status TEXT
);

CREATE TABLE Maintenance (
    maintenance_id INTEGER PRIMARY KEY,
    asset_id INTEGER,
    maintenance_date DATE,
    cost REAL,
    status TEXT,
    FOREIGN KEY(asset_id) REFERENCES Assets(asset_id)
);

CREATE TABLE Inspections (
    inspection_id INTEGER PRIMARY KEY,
    asset_id INTEGER,
    inspection_date DATE,
    result TEXT,
    FOREIGN KEY(asset_id) REFERENCES Assets(asset_id)
);

CREATE TABLE Outages (
    outage_id INTEGER PRIMARY KEY,
    asset_id INTEGER,
    start_time DATE,
    end_time DATE,
    cause TEXT,
    FOREIGN KEY(asset_id) REFERENCES Assets(asset_id)
);
""")

In [3]:
# GENERATE SYNTHETIC DATA

asset_types = ["Transformer", "Cable", "Pole", "Substation"]
locations = ["North Zone", "South Zone", "East Zone", "West Zone"]

for i in range(1, 51):
    cursor.execute("""
    INSERT INTO Assets VALUES (?, ?, ?, ?, ?)
    """, (
        i,
        random.choice(asset_types),
        random.choice(locations),
        (datetime.now() - timedelta(days=random.randint(100, 2000))).date(),
        random.choice(["Active", "Inactive", "Maintenance"])
    ))

# Maintenance data
for i in range(1, 101):
    cursor.execute("""
    INSERT INTO Maintenance VALUES (?, ?, ?, ?, ?)
    """, (
        i,
        random.randint(1, 50),
        (datetime.now() - timedelta(days=random.randint(1, 365))).date(),
        round(random.uniform(100, 5000), 2),
        random.choice(["Completed", "Pending"])
    ))

# Inspections
for i in range(1, 101):
    cursor.execute("""
    INSERT INTO Inspections VALUES (?, ?, ?, ?)
    """, (
        i,
        random.randint(1, 50),
        (datetime.now() - timedelta(days=random.randint(1, 365))).date(),
        random.choice(["Pass", "Fail"])
    ))

# Outages
for i in range(1, 51):
    start = datetime.now() - timedelta(days=random.randint(1, 200))
    end = start + timedelta(hours=random.randint(1, 72))

    cursor.execute("""
    INSERT INTO Outages VALUES (?, ?, ?, ?, ?)
    """, (
        i,
        random.randint(1, 50),
        start,
        end,
        random.choice(["Weather", "Failure", "Overload"])
    ))

conn.commit()

/tmp/ipykernel_4661/2284066372.py:7: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""
/tmp/ipykernel_4661/2284066372.py:19: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""
/tmp/ipykernel_4661/2284066372.py:31: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""
/tmp/ipykernel_4661/2284066372.py:45: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [4]:
# ANALYTICS QUERIES

print("\nTop 5 Assets with Most Maintenance:")
df1 = pd.read_sql_query("""
SELECT asset_id, COUNT(*) as maintenance_count
FROM Maintenance
GROUP BY asset_id
ORDER BY maintenance_count DESC
LIMIT 5
""", conn)
print(df1)

print("\nFailure Rate by Asset:")
df2 = pd.read_sql_query("""
SELECT asset_id,
SUM(CASE WHEN result='Fail' THEN 1 ELSE 0 END)*1.0/COUNT(*) as failure_rate
FROM Inspections
GROUP BY asset_id
ORDER BY failure_rate DESC
LIMIT 5
""", conn)
print(df2)

print("\nOutage Duration per Asset:")
df3 = pd.read_sql_query("""
SELECT asset_id,
AVG(julianday(end_time) - julianday(start_time)) as avg_outage_days
FROM Outages
GROUP BY asset_id
ORDER BY avg_outage_days DESC
LIMIT 5
""", conn)
print(df3)

conn.close()


Top 5 Assets with Most Maintenance:
   asset_id  maintenance_count
0        19                  7
1        44                  4
2        43                  4
3        30                  4
4        11                  4

Failure Rate by Asset:
   asset_id  failure_rate
0        45          1.00
1        43          1.00
2        30          1.00
3         4          1.00
4         1          0.75

Outage Duration per Asset:
   asset_id  avg_outage_days
0        36         2.958333
1         8         2.833333
2         1         2.708333
3        49         2.583333
4        29         2.333333
